In [7]:
from src.data import get_dataset
import pandas as pd
# average query len
# nqueries
# nqrels test (per level)

In [ ]:
def stats_from_dataset(dataset):
    stats = {}
    dataset = get_dataset(dataset_name=dataset)
    stats |= dataset.qrels_test.groupby("relevance").size().to_dict()
    stats |= {"n_qrels": len(dataset.qrels_test)}
    stats |= {"n_topics": dataset.qrels_test["query_id"].nunique()}
    topics = dataset.topic_components(nqueries=1, ndocspos=1, ndocsneg=1)
    for component in ["title", "description", "narrative"]:
        min_words = 0
        max_words = 0
        mean_words = 0

        for text in topics[component]:
            n_words = len(text.split())
            mean_words += n_words
            if n_words < min_words or min_words == 0:
                min_words = n_words
            if n_words > max_words:
                max_words = n_words

        mean_words /= len(topics[component])

        stats |= {
            f"min_{component}_words": round(min_words),
            f"max_{component}_words": round(max_words),
            f"mean_{component}_words": round(mean_words),
        }
    return stats

In [9]:
stats_table = []
for dataset_name in ["robust", "dl19", "dl20", "longeval-C-45", "longeval-45-C", "longeval-45-45", "longeval"]:
    stats = stats_from_dataset(dataset_name)
    stats["dataset"] = dataset_name
    stats_table.append(stats)

Could not decode response: {M: 0, T: 1, O: 0}, returning relevance: 0


[topic_gen] [WARNING] (data.py:137) Could not sample Documents: No documents provided.
[topic_gen] [WARNING] (data.py:251) No query variants available for this dataset. Returning only the original query.
[topic_gen] [WARNING] (data.py:251) No query variants available for this dataset. Returning only the original query.
[topic_gen] [WARNING] (data.py:251) No query variants available for this dataset. Returning only the original query.
[topic_gen] [WARNING] (data.py:251) No query variants available for this dataset. Returning only the original query.
[topic_gen] [WARNING] (data.py:251) No query variants available for this dataset. Returning only the original query.
[topic_gen] [WARNING] (data.py:251) No query variants available for this dataset. Returning only the original query.


In [10]:
df = pd.DataFrame(stats_table)
df

,0,1,2,n_qrels,n_topics,min_title_words,max_title_words,mean_title_words,min_description_words,max_description_words,mean_description_words,min_narrative_words,max_narrative_words,mean_narrative_words,dataset,3
0,1271,1453,227.0,2951,248,1,5,3,5,62,15,11,141,40,robust,NaN
1,557,173,195.0,1000,43,2,9,5,0,0,0,0,0,0,dl19,75.0
2,683,170,90.0,1000,54,2,15,6,0,0,0,0,0,0,dl20,57.0
3,1125,1125,NaN,2250,45,1,6,3,0,0,0,0,0,0,longeval-C-45,NaN
4,486,133,29.0,648,45,1,6,3,0,0,0,0,0,0,longeval-45-C,NaN
5,750,750,NaN,1500,45,1,6,3,0,0,0,0,0,0,longeval-45-45,NaN
6,486,133,29.0,648,45,1,6,3,0,0,0,0,0,0,longeval,NaN


In [20]:
df = df[["dataset", "n_qrels", 0, 1, 2, 3, "n_topics", "min_title_words", "max_title_words", "mean_title_words", "min_description_words", "max_description_words", "mean_description_words", "min_narrative_words", "max_narrative_words", "mean_narrative_words"]]

In [22]:
# Reorganize columns with MultiIndex
# Qrels columns: 0, 1, 2, 3, n_qrels, n_topics
# Topics columns: min/max/mean for title, description, narrative

# Create a copy and reorder columns
df_formatted = df.copy()

# Set dataset as index
df_formatted = df_formatted.set_index('dataset')

# Define column groupings
qrels_cols = ['n_qrels', 0, 1, 2, 3]
topic_cols = [
    'n_topics', 'min_title_words', 'max_title_words', 'mean_title_words',
    'min_description_words', 'max_description_words', 'mean_description_words',
    'min_narrative_words', 'max_narrative_words', 'mean_narrative_words'
]

# Reorder columns
df_formatted = df_formatted[qrels_cols + topic_cols]

# Create 3-level MultiIndex columns
column_tuples = [
    ('Qrels', "", '#Qrels'),
    ('Qrels', "Relevance", '0'),
    ('Qrels', "Relevance", '1'),
    ('Qrels', "Relevance", '2'),
    ('Qrels', "Relevance", '3' ),
    ('Topics', '#Topics', ''),
    ('Topics', 'Title', 'Min'),
    ('Topics', 'Title', 'Max'),
    ('Topics', 'Title', 'Mean'),
    ('Topics', 'Description', 'Min'),
    ('Topics', 'Description', 'Max'),
    ('Topics', 'Description', 'Mean'),
    ('Topics', 'Narrative', 'Min'),
    ('Topics', 'Narrative', 'Max'),
    ('Topics', 'Narrative', 'Mean'),
]

df_formatted.columns = pd.MultiIndex.from_tuples(column_tuples)

# Fill NaN with '-' for better display
df_formatted = df_formatted.fillna('-')

# Convert float columns to int where appropriate (before they were filled with '-')
for col in df_formatted.columns:
    if col[1].startswith('Rel-') or col[1] in ['#Topics', '#Qrels']:
        df_formatted[col] = df_formatted[col].apply(
            lambda x: int(x) if x != '-' else '-')

df_formatted

Qrels                               Topics                 \
                      Relevance                    #Topics Title            
               #Qrels         0     1      2     3           Min Max Mean   
dataset                                                                     
robust           2951      1271  1453  227.0     -     248     1   5    3   
dl19             1000       557   173  195.0  75.0      43     2   9    5   
dl20             1000       683   170   90.0  57.0      54     2  15    6   
longeval-C-45    2250      1125  1125      -     -      45     1   6    3   
longeval-45-C     648       486   133   29.0     -      45     1   6    3   
longeval-45-45   1500       750   750      -     -      45     1   6    3   
longeval          648       486   133   29.0     -      45     1   6    3   

                                                         
               Description          Narrative            
                       Min Max Mean       Min  Max Mean  
dataset                                                  
robust                   5  62   15        11  141   40  
dl19                     0   0    0         0    0    0  
dl20                     0   0    0         0    0    0  
longeval-C-45            0   0    0         0    0    0  
longeval-45-C            0   0    0         0    0    0  
longeval-45-45           0   0    0         0    0    0  
longeval                 0   0    0         0    0    0

In [23]:
# Export as LaTeX for paper (spanning both columns)
latex_table = df_formatted.to_latex(
    multirow=True,
    multicolumn=True,
    escape=False,
    column_format='l' + 'r' * len(df_formatted.columns),
    caption='Dataset Statistics',
    label='tab:dataset-stats'
)

# Replace table with table* for two-column layout
latex_table = latex_table.replace(r'\begin{table}', r'\begin{table*}')
latex_table = latex_table.replace(r'\end{table}', r'\end{table*}')

print(latex_table)

\begin{table*}
\caption{Dataset Statistics}
\label{tab:dataset-stats}
\begin{tabular}{lrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{5}{r}{Qrels} & \multicolumn{10}{r}{Topics} \\
 &  & \multicolumn{4}{r}{Relevance} & #Topics & \multicolumn{3}{r}{Title} & \multicolumn{3}{r}{Description} & \multicolumn{3}{r}{Narrative} \\
 & #Qrels & 0 & 1 & 2 & 3 &  & Min & Max & Mean & Min & Max & Mean & Min & Max & Mean \\
dataset &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
robust & 2951 & 1271 & 1453 & 227.000000 & - & 248 & 1 & 5 & 3 & 5 & 62 & 15 & 11 & 141 & 40 \\
dl19 & 1000 & 557 & 173 & 195.000000 & 75.000000 & 43 & 2 & 9 & 5 & 0 & 0 & 0 & 0 & 0 & 0 \\
dl20 & 1000 & 683 & 170 & 90.000000 & 57.000000 & 54 & 2 & 15 & 6 & 0 & 0 & 0 & 0 & 0 & 0 \\
longeval-C-45 & 2250 & 1125 & 1125 & - & - & 45 & 1 & 6 & 3 & 0 & 0 & 0 & 0 & 0 & 0 \\
longeval-45-C & 648 & 486 & 133 & 29.000000 & - & 45 & 1 & 6 & 3 & 0 & 0 & 0 & 0 & 0 & 0 \\
longeval-45-45 & 1500 & 750 & 750 & - & - & 45 & 1 & 6 & 3 & 0 & 0 